# Step 4 — Modeling (Regression & Classification)

**Dataset:** `uk_housing_2010_2017_processed.csv`  
**Targets:**  
- Regression → `log_price`  
- Classification → `is_high_value` (top 20% by local authority/year)  

**Models:**  
- Regression: Linear Regression, Random Forest Regressor, Gradient Boosting Regressor  
- Classification: Logistic Regression, Random Forest Classifier, Gradient Boosting Classifier  

**Outputs saved to:**
- `reports/metrics/` — CSV tables with metrics  
- `models/` — fitted pipelines via `joblib`


In [3]:
import sys
from pathlib import Path

# Add project root to sys.path (so config.py is found)
root = Path().resolve().parent  # one level up from notebooks/
sys.path.append(str(root))

from config import RAW_DIR, INTERIM_DIR
import pandas as pd

print("RAW_DIR:", RAW_DIR)
print("INTERIM_DIR:", INTERIM_DIR)


RAW_DIR: C:\DataProjects\uh-ds-housing-data\data\raw
INTERIM_DIR: C:\DataProjects\uh-ds-housing-data\data\interim


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from config import PROCESSED_DIR
from models.random_forest_regressor import build_pipeline as build_rf
from models.gradient_boosting_regressor import build_pipeline as build_gbr

# 1) Load processed data (time-aware)
data_path = PROCESSED_DIR / "uk_housing_2010_2017_processed.csv"
df = pd.read_csv(data_path, parse_dates=["date_of_transfer"], low_memory=False)

# Ensure time order (important for TimeSeriesSplit)
df = df.sort_values("date_of_transfer").reset_index(drop=True)

# Use a manageable sample for CV + tuning
df_sample = df.sample(50_000, random_state=42).copy()

features = ["year", "month", "property_type", "county", "district", "town_city"]
target = "log_price"

X = df_sample[features]
y = df_sample[target]

num_features = ["year", "month"]
cat_features = [f for f in features if f not in num_features]

# Hold-out split without shuffling (time-aware)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)


Train size: (40000, 6) Test size: (10000, 6)


In [5]:
def evaluate_model(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds, squared=False)
    r2 = r2_score(y_test, preds)

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }, preds

rf_model = build_rf(num_features, cat_features)
gbr_model = build_gbr(num_features, cat_features)

rf_results, rf_preds = evaluate_model(rf_model, X_train, y_train, X_test, y_test, "Random Forest")
gbr_results, gbr_preds = evaluate_model(gbr_model, X_train, y_train, X_test, y_test, "Gradient Boosting")

results_df = pd.DataFrame([rf_results, gbr_results]).sort_values("R2", ascending=False)
results_df


TypeError: got an unexpected keyword argument 'squared'

## Cell 3 — Plot RF vs GBR (Errors + Fit)


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(results_df["Model"], results_df["RMSE"])
plt.title("Model Comparison (Lower RMSE is Better)")
plt.ylabel("RMSE")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Actual vs Predicted plot for both
plt.figure(figsize=(6, 6))
plt.scatter(y_test, rf_preds, s=10, alpha=0.3, label="RF")
plt.scatter(y_test, gbr_preds, s=10, alpha=0.3, label="GBR")
minv, maxv = y_test.min(), y_test.max()
plt.plot([minv, maxv], [minv, maxv], "r--", linewidth=2)
plt.title("Actual vs Predicted (Log Price)")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Cell 4 — Tuning Random Forest (Safe Grid)


In [6]:
tscv = TimeSeriesSplit(n_splits=5)

rf_tune = build_rf(num_features, cat_features)

rf_param_grid = {
    "rf__n_estimators": [100, 200],
    "rf__max_depth": [15, 25],
    "rf__min_samples_split": [2, 10]
}

rf_search = GridSearchCV(
    rf_tune,
    rf_param_grid,
    cv=tscv,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

print("Best RF Params:", rf_search.best_params_)
print("Best RF CV R2:", rf_search.best_score_)


Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best RF Params: {'rf__max_depth': 25, 'rf__min_samples_split': 10, 'rf__n_estimators': 200}
Best RF CV R2: 0.4910850604077218


## 3) Train/test split
Use a fixed random seed for reproducibility.


In [5]:
# === Cell 4: Split ===
X = dfm[num_features + cat_features]
y_reg = dfm[reg_target]
y_cls = dfm[cls_target].astype(int)

X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42, shuffle=True
)
# keep the same indices for classification to have aligned splits
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, shuffle=True
)

print("Train:", X_train.shape, "| Test:", X_test.shape)


Train: (4755872, 8) | Test: (1188969, 8)


## 4) Preprocessing pipeline
- Impute numeric with median + scale  
- Impute categorical with most-frequent + one-hot encode


In [6]:
# === Cell 5: Preprocessor ===
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features)
])

print("Preprocessor ready.")


Preprocessor ready.


## 5) Regression models
We’ll train three levels:  
- **Baseline:** Linear Regression  
- **Medium:** Random Forest Regressor  
- **Advanced:** Gradient Boosting Regressor  

Metrics:
- \( \text{MAE} = \frac{1}{n}\sum |y-\hat{y}| \)  
- \( \text{RMSE} = \sqrt{\frac{1}{n}\sum (y-\hat{y})^2} \)  
- \( R^2 = 1 - \frac{\sum (y-\hat{y})^2}{\sum (y-\bar{y})^2} \)


In [ ]:
# === Cell 6: Fit regression models ===
reg_results = {}

def eval_reg(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    reg_results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}

# 6.1 Linear Regression
reg_lin = Pipeline([
    ("prep", preprocessor),
    ("model", LinearRegression())
])
reg_lin.fit(X_train, y_train_reg)
pred_lin = reg_lin.predict(X_test)
eval_reg(y_test_reg, pred_lin, "LinearRegression")

# 6.2 Random Forest (with light tuning)
reg_rf = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200, max_depth=None, n_jobs=-1, random_state=42
    ))
])
reg_rf.fit(X_train, y_train_reg)
pred_rf = reg_rf.predict(X_test)
eval_reg(y_test_reg, pred_rf, "RandomForestRegressor")

# 6.3 Gradient Boosting (light tuning)
reg_gb = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42
    ))
])
reg_gb.fit(X_train, y_train_reg)
pred_gb = reg_gb.predict(X_test)
eval_reg(y_test_reg, pred_gb, "GradientBoostingRegressor")

pd.DataFrame(reg_results).T.sort_values("RMSE")


C:\Users\BALA\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


## 6) Classification models
We’ll train:
- **Baseline:** Logistic Regression (L2, liblinear)  
- **Medium:** Random Forest Classifier  
- **Advanced:** Gradient Boosting Classifier

Metrics:
- Accuracy \( \frac{\text{TP+TN}}{\text{Total}} \)  
- F1 \( = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision}+\text{Recall}} \)  
- ROC–AUC (threshold-free ranking quality)


In [ ]:
# === Cell 7: Fit classification models ===
cls_results = {}

def eval_cls(y_true, y_proba, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_proba)
    except Exception:
        auc = np.nan
    cls_results[name] = {"Accuracy": acc, "F1": f1, "ROC_AUC": auc}

# 7.1 Logistic Regression
cls_log = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=200, solver="liblinear"))
])
cls_log.fit(X_train_cls, y_train_cls)
proba_log = cls_log.predict_proba(X_test_cls)[:, 1]
pred_log = (proba_log >= 0.5).astype(int)
eval_cls(y_test_cls, proba_log, pred_log, "LogisticRegression")

# 7.2 Random Forest Classifier
cls_rf = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300, max_depth=None, n_jobs=-1, random_state=42
    ))
])
cls_rf.fit(X_train_cls, y_train_cls)
proba_rf = cls_rf.predict_proba(X_test_cls)[:, 1]
pred_rf = (proba_rf >= 0.5).astype(int)
eval_cls(y_test_cls, proba_rf, pred_rf, "RandomForestClassifier")

# 7.3 Gradient Boosting Classifier
cls_gb = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42
    ))
])
cls_gb.fit(X_train_cls, y_train_cls)
proba_gb = cls_gb.predict_proba(X_test_cls)[:, 1]
pred_gb = (proba_gb >= 0.5).astype(int)
eval_cls(y_test_cls, proba_gb, pred_gb, "GradientBoostingClassifier")

pd.DataFrame(cls_results).T.sort_values("ROC_AUC", ascending=False)


## 7) Save models and metrics
Pipelines include preprocessing, so they’re **ready for inference** on raw feature columns.


In [ ]:
# === Cell 8: Save artifacts ===
joblib.dump(reg_lin, MODELS_DIR / "reg_linear.joblib")
joblib.dump(reg_rf, MODELS_DIR / "reg_random_forest.joblib")
joblib.dump(reg_gb, MODELS_DIR / "reg_gradient_boosting.joblib")

joblib.dump(cls_log, MODELS_DIR / "cls_logistic.joblib")
joblib.dump(cls_rf, MODELS_DIR / "cls_random_forest.joblib")
joblib.dump(cls_gb, MODELS_DIR / "cls_gradient_boosting.joblib")

pd.DataFrame(reg_results).T.to_csv(METRICS_DIR / "regression_metrics.csv")
pd.DataFrame(cls_results).T.to_csv(METRICS_DIR / "classification_metrics.csv")

print("Saved models →", MODELS_DIR)
print("Saved metrics →", METRICS_DIR)


In [ ]:
## 8) Quick diagnostic plots (optional)
- Residual distribution for the best regression model  
- ROC curve for the best classifier


In [ ]:
# === Cell 9: Quick plots (simple, optional) ===
# Residuals for the gradient boosting regressor (often best)
pred = reg_gb.predict(X_test)
resid = y_test_reg - pred

plt.figure(figsize=(6,4))
plt.hist(resid, bins=100)
plt.title("Residuals (GBR)")
plt.xlabel("y_true - y_pred")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# ROC curve for the best classifier (pick GB here)
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test_cls, proba_gb)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label="GB Classifier")
plt.plot([0,1],[0,1],"--", label="Chance")
plt.title("ROC Curve")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.legend()
plt.tight_layout()
plt.show()
